# Context Engineering

**Context engineering means choosing what information the model should see for its next step.**

A model can receive instructions, conversation history, tool results, retrieved documents, and saved notes. Giving it everything is tempting, but irrelevant material can hide the information that matters. The goal is therefore not the largest possible context; it is the smallest context that contains everything needed for the current task.


## Mental model: pack a small work desk

Think of the context window as a work desk. The model can use only what is currently on the desk.

- Put permanent instructions and the current task on the desk.
- Add only the documents needed for the current step.
- Move useful long-term facts into notes.
- Remove old tool output and irrelevant conversation.
- Summarize old discussion when the desk becomes crowded.

A larger desk helps, but a desk covered in unrelated papers is still difficult to use.


## What can be in context?

| Part | Example | Usually keep? |
|---|---|---|
| Instructions | `Do not change public APIs` | Yes |
| Current task | `Fix the login timeout` | Yes |
| Relevant source | `auth.py` | Only when needed |
| Saved memory | `The project uses PostgreSQL` | If relevant |
| Recent history | The last few decisions | Usually |
| Old tool output | A long test log from 20 steps ago | Usually remove |

For every item, ask: **Will this help the model make its next decision?** If not, leave it out.


## Four practical techniques

### 1. Load information only when needed

Keep a file path or document id, then fetch the content when the task requires it. This is called **just-in-time retrieval**. For example, an agent fixing login code may need `auth.py`, but it probably does not need the billing schema.

### 2. Remove replaceable output

Large search results, file listings, and test logs often become useless after the model has acted on them. Remove them when they are old. They can be fetched again if necessary.

### 3. Summarize old history

When a conversation becomes long, replace older messages with a short summary. This is **compaction**. A useful summary preserves:

- decisions already made,
- important constraints,
- completed work,
- unresolved problems, and
- the next step.

### 4. Save durable facts outside the conversation

Some information must survive even when history is compacted: project conventions, user preferences, or architectural decisions. Store these facts in a small memory or notes file and reload them when relevant.


## A small example

Suppose an agent must fix a login bug. Six pieces of information are available, but the prompt has room for only four. The agent should always keep required items, then prefer optional items that are relevant to login.


In [ ]:
# Token counts are simplified estimates for this demonstration.
context_items = [
    {"name": "project instructions", "tokens": 200, "relevance": 10, "required": True},
    {"name": "current login task", "tokens": 100, "relevance": 10, "required": True},
    {"name": "auth.py", "tokens": 500, "relevance": 9, "required": False},
    {"name": "recent login test failure", "tokens": 300, "relevance": 8, "required": False},
    {"name": "old search output", "tokens": 700, "relevance": 2, "required": False},
    {"name": "billing documentation", "tokens": 600, "relevance": 1, "required": False},
]


def build_context(items, budget):
    """Keep required items, then add the most relevant items that fit."""
    selected = [item for item in items if item["required"]]
    used = sum(item["tokens"] for item in selected)

    optional = sorted(
        (item for item in items if not item["required"]),
        key=lambda item: item["relevance"],
        reverse=True,
    )

    for item in optional:
        if used + item["tokens"] <= budget:
            selected.append(item)
            used += item["tokens"]

    return selected, used


selected, used = build_context(context_items, budget=1_200)
print(f"Selected {used} of 1,200 available tokens:")
for item in selected:
    print(f"- {item['name']}")


The example keeps the instructions, task, login code, and relevant test failure. It leaves out the old search output and billing documentation. Those items are not bad; they are simply unhelpful for the next step.

This scoring rule is deliberately simple. A real system may use search, metadata, model-based ranking, or application rules to decide relevance. The core decision remains the same: **required first, relevant next, noise out**.


## Common mistakes

- **Adding everything:** more text can distract the model from the task.
- **Removing important decisions:** a poor summary can make the agent repeat work.
- **Keeping large old tool results:** retain the conclusion, not every raw line.
- **Loading stale information:** fetch changing data close to when it is used.
- **Trusting the maximum window size:** fitting inside the limit does not mean every detail will receive equal attention.


## Practice

1. Change the budget from `1_200` to `800`. Which useful item no longer fits?
2. Add a relevant saved note using 150 tokens. What should the system remove to make room?
3. Write a five-line compaction summary containing a decision, constraint, completed action, open problem, and next step.

## Before you ship

- [ ] Required instructions and the current task are always present.
- [ ] Large or changing documents are fetched only when needed.
- [ ] Old tool output is removed when it can be fetched again.
- [ ] Compaction preserves decisions, constraints, open problems, and next steps.
- [ ] Durable facts live outside disposable conversation history.
- [ ] The system is tested with long conversations, not only short examples.
